## Test function to combine TCRs
Try to get a way to combine two separate datasets so it can feed into the `standardize_to_sample_format` function

## 

In [1]:
import sys
sys.path.insert(0, "..")
sys.path.insert(0, "../tools")

import generate_TCRAFT_oligopool as gt
import pandas as pd
from config import ALL_DATA_DIR

In [2]:
# load csvs
AS_df = pd.read_csv(ALL_DATA_DIR / "AS_Top_96_TCRs.csv")
Ctrl_df = pd.read_csv(ALL_DATA_DIR / "Positive_Ctrl_TCRs.csv")

In [3]:
def standardize_to_sample_format(df, select=None, output_dir=None, filename="standardized_TCRs.csv"):
    """Convert a df in AS_top_96 or positive_ctrls format into sample_df format.

    select: optionally, an integer number of TCRs to randomly select from the result.
    output_dir: optionally, a directory to save the standardized df to as a CSV.
                The directory is created if it doesn't already exist.
    filename: name of the CSV file to write into output_dir (ignored if output_dir is None).
    """
    df = df.copy()
    df.columns = df.columns.str.replace("﻿", "", regex=False).str.strip()

    # AS_top_96 uses short lowercase column names; rename to sample_df's convention.
    # positive_ctrls already uses sample_df's column names (plus extra columns).
    column_map = {
        "CLONE_ID": "Name",
        "name": "Name",
        "va": "V_alpha",
        "vb": "V_beta",
        "cdr3a": "CDR3_alpha",
        "cdr3b": "CDR3_beta",
        "ja": "J_alpha",
        "jb": "J_beta",
    }
    df = df.rename(columns=column_map)

    # A concatenated input can carry both conventions at once (e.g. va and V_alpha,
    # or CLONE_ID and Name), so renaming collapses them onto duplicate labels. Merge
    # each set of duplicates into a single column, keeping the first non-null value
    # per row. Left alone, the duplicates would also make df[sample_cols] below
    # return every matching column rather than one per name.
    if df.columns.duplicated().any():
        df = pd.concat(
            [df.loc[:, df.columns == col].bfill(axis=1).iloc[:, 0].rename(col)
             for col in df.columns.unique()],
            axis=1,
        )

    sample_cols = ["Name", "V_alpha", "V_beta", "CDR3_alpha", "CDR3_beta", "J_alpha", "J_beta"]
    df = df[sample_cols].dropna(how="all").reset_index(drop=True)

    if select is not None:
        df = df.sample(n=select).reset_index(drop=True)

    if output_dir is not None:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        df.to_csv(output_dir / filename, index=False)

    return df

In [5]:
df = pd.concat([AS_df, Ctrl_df], ignore_index=True)

In [6]:
standardize_to_sample_format(df)

,Name,V_alpha,V_beta,CDR3_alpha,CDR3_beta,J_alpha,J_beta
0,AS_Clone0319,TRAV1-2,TRBV6-2,CAFWAGGFKTIF,CASSYGDISYGYTF,TRAJ9,TRBJ1-2
1,AS_Clone0819,TRAV12-2,TRBV28,CAVNIHSGYALNF,CASSLLAGGNNEQFF,TRAJ41,TRBJ2-1
2,AS_Clone0907,TRAV12-2,TRBV28,CAAAEGNTGFQKLVF,CASSLWTAGDNQPQHF,TRAJ8,TRBJ1-5
3,AS_Clone3389,TRAV29/DV5,TRBV3-1,CAASAFMSSYGKLTF,CASSQEPGTSTYEQYF,TRAJ52,TRBJ2-7
4,AS_Clone1222,TRAV13-1,TRBV6-1,CAASIRGGSYIPTF,CASSASGSTDTQYF,TRAJ6,TRBJ2-3
...,...,...,...,...,...,...,...
94,AS_Clone2977,TRAV26-2,TRBV29-1,CILRGSYNQGGKLIF,CSVELGTVLTGANTEAFF,TRAJ23,TRBJ1-1
95,AS_Clone5095,TRAV8-6,TRBV29-1,CAVSGSLETSYDKVIF,CSVRVEGYEQYF,TRAJ50,TRBJ2-7
96,gp100,TRAV21,TRBV7-3,CAVLSSGGSNYKLTF,CASSFIGGTDTQYF,TRAJ53,TRBJ2-3
97,gp100,TRAV17,TRBV19,CATDGDTPLVF,CASSIGGPYEQYF,TRAJ29,TRBJ2-7


In [ ]:
from pathlib import Path


def standardize_to_sample_format(df, select=None, output_dir=None, filename="standardized_TCRs.csv"):
    """Convert df(s) in AS_top_96 or positive_ctrls format into sample_df format.

    df: a single df, or a list/tuple of dfs which are concatenated before being
        standardized (so select/output_dir apply to the combined result).
    select: optionally, an integer number of TCRs to randomly select from the result.
            Empty rows dropped during standardization are subtracted from this
            count, since select is typically derived from len() of the raw input.
    output_dir: optionally, a directory to save the standardized df to as a CSV.
                The directory is created if it doesn't already exist.
    filename: name of the CSV file to write into output_dir (ignored if output_dir is None).
    """
    if isinstance(df, (list, tuple)):
        if len(df) == 0:
            raise ValueError("df list is empty; pass at least one dataframe.")
        df = pd.concat([d.copy() for d in df], ignore_index=True)
    else:
        df = df.copy()

    df.columns = df.columns.str.replace("﻿", "", regex=False).str.strip()

    # AS_top_96 uses short lowercase column names; rename to sample_df's convention.
    # positive_ctrls already uses sample_df's column names (plus extra columns).
    column_map = {
        "CLONE_ID": "Name",
        "name": "Name",
        "va": "V_alpha",
        "vb": "V_beta",
        "cdr3a": "CDR3_alpha",
        "cdr3b": "CDR3_beta",
        "ja": "J_alpha",
        "jb": "J_beta",
    }
    df = df.rename(columns=column_map)

    # A concatenated input can carry both conventions at once (e.g. va and V_alpha,
    # or CLONE_ID and Name), so renaming collapses them onto duplicate labels. Merge
    # each set of duplicates into a single column, keeping the first non-null value
    # per row. Left alone, the duplicates would also make df[sample_cols] below
    # return every matching column rather than one per name.
    if df.columns.duplicated().any():
        df = pd.concat(
            [df.loc[:, df.columns == col].bfill(axis=1).iloc[:, 0].rename(col)
             for col in df.columns.unique()],
            axis=1,
        )

    sample_cols = ["Name", "V_alpha", "V_beta", "CDR3_alpha", "CDR3_beta", "J_alpha", "J_beta"]

    # Some source CSVs carry blank trailing rows (AS_TCRs_All_TCRs.csv has 489),
    # which read_csv turns into all-NA rows. Drop them, report the count, and keep
    # it so select can be corrected below.
    n_before = len(df)
    df = df[sample_cols].dropna(how="all").reset_index(drop=True)
    n_dropped = n_before - len(df)
    if n_dropped:
        print(f"NA detected: dropped {n_dropped} empty row(s) of {n_before}; {len(df)} TCRs remain.")

    if select is not None:
        if n_dropped:
            select -= n_dropped
            print(f"select adjusted by -{n_dropped} to {select}.")
        if select <= 0:
            raise ValueError(
                f"select is {select} after subtracting {n_dropped} dropped empty row(s); "
                "pass a select count based on the raw row count, or None to keep all rows."
            )
        if select > len(df):
            raise ValueError(f"select={select} exceeds the {len(df)} TCRs available.")
        df = df.sample(n=select).reset_index(drop=True)

    if output_dir is not None:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        df.to_csv(output_dir / filename, index=False)

    return df


In [9]:
# list input: concatenate then standardize
standardize_to_sample_format(Ctrl_df)


,Name,V_alpha,V_beta,CDR3_alpha,CDR3_beta,J_alpha,J_beta
0,gp100,TRAV21,TRBV7-3,CAVLSSGGSNYKLTF,CASSFIGGTDTQYF,TRAJ53,TRBJ2-3
1,gp100,TRAV17,TRBV19,CATDGDTPLVF,CASSIGGPYEQYF,TRAJ29,TRBJ2-7
2,MART-1,TRAV12-2,TRBV6-4,CAVNFGGGKLIF,CASSLSFGTEAFF,TRAJ23,TRBJ1-1


### Validating also the oligo pools

In [4]:
PoolA = pd.read_csv(ALL_DATA_DIR / "01_AS_96_pilot" / "01_PoolA_IDT_oligos.csv")

In [ ]:
Names = PoolA["Name"]


PoolA_TCRs = AS_df[AS_df["CLONE_ID"].isin(Names)]
PoolA_TCRs


,va,ja,cdr3a,vb,jb,cdr3b,CLONE_ID,nndists_tcr,clusters_tcr_names
0,TRAV1-2,TRAJ9,CAFWAGGFKTIF,TRBV6-2,TRBJ1-2,CASSYGDISYGYTF,AS_Clone0319,96.600000,10_AV1
3,TRAV29/DV5,TRAJ52,CAASAFMSSYGKLTF,TRBV3-1,TRBJ2-7,CASSQEPGTSTYEQYF,AS_Clone3389,110.777778,103_bv4
4,TRAV13-1,TRAJ6,CAASIRGGSYIPTF,TRBV6-1,TRBJ2-3,CASSASGSTDTQYF,AS_Clone1222,113.266667,108_AV13
5,TRAV12-3,TRAJ40,CAFDPGTYKYIF,TRBV6-1,TRBJ1-5,CASSDEGPQHF,AS_Clone0984,33.355556,109_BV6
8,TRAV1-2,TRAJ20,CAVNNDYKLSF,TRBV6-4,TRBJ2-2,CASSQGGENTGELFF,AS_Clone0164,54.133333,29_BV6
10,TRAV21,TRAJ15,CAVREAGTALIF,TRBV6-1,TRBJ2-1,CASSESTRSSYNEQFF,AS_Clone2338,120.044444,3_BV6
12,TRAV25,TRAJ10,CAGITGGGNKLTF,TRBV6-5,TRBJ2-1,CASGGTGPYNEQFF,AS_Clone2814,88.200000,34_av30
17,TRAV29/DV5,TRAJ29,CAASLYSGNTPLVF,TRBV2,TRBJ1-2,CASSEARGVNYGYTF,AS_Clone3256,100.688889,19_av29
19,TRAV38-2/DV8,TRAJ53,CAYRGGSNYKLTF,TRBV7-8,TRBJ1-6,CASSLLQGPSSPLHF,AS_Clone4060,36.333333,1_AV38
20,TRAV38-2/DV8,TRAJ30,CAYRSGDDKIIF,TRBV2,TRBJ1-6,CASSEPAALNSPLHF,AS_Clone3968,87.311111,1_AV38


the pools are separated by TRBV for vector pool 1A and 1B BUT are not separated by TRAV for input 
- I assume then you will have to place the vector pool products of 1A into both 2A and 2B pools.
- so in total you will have 4 vector pools at the end
- 1A-2A, 1A-2B, 1B-2A, 1B-2B